# YouTube-bot One-Tap Mobile Video Builder

Open on a phone, tap **Runtime → Run all**, and wait for the ZIP download. This notebook is pinned to tested commit `63a4d57d4a90a50707044447fe810cbaec4f2f41`.

Default output: offline `slides` MP4. No keys, computer, GitHub token, or YouTube credentials required.

In [ ]:
#@title Set topic and output
TOPIC = 'Why the sky changes color' #@param {type:'string'}
EPISODE_ID = 'mobile-one-tap-001' #@param {type:'string'}
VIDEO_MODE = 'slides' #@param ['slides', 'ken-burns', 'shorts-vertical', 'audiogram']
PINNED_COMMIT = '63a4d57d4a90a50707044447fe810cbaec4f2f41'
print(f'Topic: {TOPIC} | Mode: {VIDEO_MODE}')

In [ ]:
#@title Build and download
import os, shutil, subprocess
from pathlib import Path
from google.colab import files
subprocess.run(['rm','-rf','YouTube-bot'], check=True)
subprocess.run(['git','clone','-q','https://github.com/studio1111/YouTube-bot.git'], check=True)
os.chdir('YouTube-bot')
subprocess.run(['git','checkout','-q',PINNED_COMMIT], check=True)
subprocess.run(['python','-m','pip','install','-q','-e','.'], check=True)
if shutil.which('ffmpeg') is None:
    subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
subprocess.run(['python','-m','yt_agent.cli','run','--topic',TOPIC,'--episode-id',EPISODE_ID,'--runtime-dir','runtime','--video-mode',VIDEO_MODE], check=True)
episode = Path('runtime') / 'episodes' / EPISODE_ID / 'v1'
video = episode / 'video.mp4'
if not video.exists():
    error = episode / 'render_error.log'
    print(error.read_text() if error.exists() else 'No MP4 and no diagnostic log.')
    raise RuntimeError('Video render failed. See the diagnostic above.')
bundle = Path(f'{EPISODE_ID}-video-package')
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(episode, bundle)
archive = shutil.make_archive(str(bundle), 'zip', root_dir=bundle)
files.download(archive)
print(f'Downloaded: {archive}')